<a href="https://colab.research.google.com/github/profcomff/chatbot-mark-api/blob/main/notebooks/Copy_of_Create_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
test_e5_no_prefix = True
test_e5_with_prefix = True

In [ ]:
# !git clone https://github.com/profcomff/chatbot-mark-api.git
!git clone --branch dev_fedor https://github.com/profcomff/chatbot-mark-api.git

Cloning into 'chatbot-mark-api'...
remote: Enumerating objects: 284, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 284 (delta 48), reused 84 (delta 29), pack-reused 169 (from 1)
Receiving objects: 100% (284/284), 3.07 MiB | 6.84 MiB/s, done.
Resolving deltas: 100% (100/100), done.


# Библиотеки


In [ ]:
!pip install langchain transformers sentence-transformers -q
!pip install -U langchain-community -q
!pip install -qU "langchain-chroma>=0.1.2" -q
!pip install langchain_huggingface -q
!pip install rank_bm25 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [ ]:
# Core Python libraries
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import pickle

from langchain_core.embeddings import Embeddings
from transformers import XLMRobertaTokenizer, XLMRobertaModel
import torch

# Torch and Transformers
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers import (
    AutoModelForSequenceClassification,
    AutoModel,
    AutoTokenizer,
    XLMRobertaTokenizer,
    XLMRobertaModel,
)

# NLTK for text preprocessing
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

# Sentence Transformers
from sentence_transformers import CrossEncoder

# LangChain - documents and embeddings
from langchain_core.documents import Document
from langchain.schema import Document
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore


# LangChain - loading, splitting, and vector stores
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma

from langchain.embeddings import HuggingFaceEmbeddings #this?
from langchain_huggingface import HuggingFaceEmbeddings #or this?

# LangChain - retrieval and reranking
from langchain.retrievers import (
    ContextualCompressionRetriever,
    EnsembleRetriever
)
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.retrievers import BM25Retriever


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# Функции/классы

In [ ]:
import sys
sys.path.append("/content/chatbot-mark-api")

from nn.search import get_context, preprocess, E5LangChainEmbedder

In [ ]:
def safe_add_documents(vector_store, chunks, chroma_batch_size=1000):
    with tqdm(total=len(chunks), desc="Добавление в Chroma", unit="doc") as pbar:
        for i in range(0, len(chunks), chroma_batch_size):
            try:
                batch = chunks[i:i+chroma_batch_size]
                vector_store.add_documents(batch)
                pbar.update(len(batch))
            except Exception as e:
                if "Batch size" in str(e) and "greater than max" in str(e):
                    new_size = chroma_batch_size // 2
                    print(f"Ошибка: {e}. Уменьшаю размер батча до {new_size}")
                    return safe_add_documents(vector_store, chunks[i:], new_size)
                raise
            finally:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    print("Все документы успешно добавлены!")

# 0. Загрузка контекстов / скачивание модели


In [ ]:
answers = pd.read_excel('/content/chatbot-mark-api/file/database_v2.xlsx')

display(answers.answer[0]) #debug
display(answers.head(2))

'Карта зачет. https://vk.com/wall-24234717_22977\nЭто ваш профсоюзный билет. С помощью этой карты вы можете получать скидки у полезных для студентов популярных брендов, участвовать в конкурсах и розыгрышах, а также посещать концерты и мероприятия. \nПолный перечень скидок есть в статье: vk.cc/bYSCNw.'

,Unnamed: 0,topic_name,answer,id
0,0,Карта зачет,Карта зачет. https://vk.com/wall-24234717_2297...,0
1,1,Как вступить в профсоюз,Как вступить в профсоюз? Чтобы вступить в Проф...,1


link to model in HuggingFace [e5-base-en-ru](https://huggingface.co/d0rj/e5-base-en-ru)

In [ ]:
tokenizer = XLMRobertaTokenizer.from_pretrained("d0rj/e5-base-en-ru", use_cache=False)
search_model = XLMRobertaModel.from_pretrained("d0rj/e5-base-en-ru", use_cache=False)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/471 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/529M [00:00<?, ?B/s]

# 1. Создание БД c помощью e5 без префиксов

In [ ]:
if test_e5_no_prefix:
    # Подготовка документов из ответов
    all_chunks = []

    for answer, topic_name in zip(answers['answer'], answers['topic_name']):
        all_chunks.append(Document(
            page_content=answer,
            metadata={
                "source": topic_name
            }
        ))

    # Инициализация эмбеддера E5 (без префиксов)
    embedder = E5LangChainEmbedder(
        tokenizer=tokenizer,
        model=search_model,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        add_prefix=False,  # Важно: не добавляем префиксы
        disable_tqdm=False,
    )

    # Создание или загрузка векторного хранилища Chroma
    vector_store = Chroma(
        collection_name="docs",
        embedding_function=embedder,
        persist_directory="./chroma_db_without-prefix"  # Путь к локальной БД
    )

    # Безопасное добавление документов в векторное хранилище
    safe_add_documents(vector_store, all_chunks)

Добавление в Chroma:   0%|          | 0/105 [00:00<?, ?doc/s]


Вычисление эмбеддингов: 100%|██████████| 14/14 [01:02<00:00,  4.46s/batch]


Все документы успешно добавлены!


In [ ]:
if test_e5_no_prefix:
    !zip -r chroma_db_without-prefix.zip chroma_db_without-prefix/

  adding: chroma_db_without-prefix/ (stored 0%)
  adding: chroma_db_without-prefix/b07a06b9-d125-44a5-ad6d-0e9b8fde2c36/ (stored 0%)
  adding: chroma_db_without-prefix/b07a06b9-d125-44a5-ad6d-0e9b8fde2c36/link_lists.bin (stored 0%)
  adding: chroma_db_without-prefix/b07a06b9-d125-44a5-ad6d-0e9b8fde2c36/header.bin (deflated 61%)
  adding: chroma_db_without-prefix/b07a06b9-d125-44a5-ad6d-0e9b8fde2c36/length.bin (deflated 28%)
  adding: chroma_db_without-prefix/b07a06b9-d125-44a5-ad6d-0e9b8fde2c36/data_level0.bin (deflated 100%)
  adding: chroma_db_without-prefix/chroma.sqlite3 (deflated 52%)


# 1. Создание БД c помощью e5 без префиксов

In [ ]:
if test_e5_with_prefix:
    # Подготовка документов из ответов
    all_chunks = []

    for answer, topic_name in zip(answers['answer'], answers['topic_name']):
        all_chunks.append(Document(
            page_content=answer,
            metadata={
                "source": topic_name
            }
        ))

    # Инициализация эмбеддера E5
    embedder = E5LangChainEmbedder(
        tokenizer=tokenizer,
        model=search_model,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        add_prefix=True,  #!!!
        disable_tqdm=False,
    )

    # Создание или загрузка векторного хранилища Chroma
    vector_store = Chroma(
        collection_name="docs",
        embedding_function=embedder,
        persist_directory="./chroma_db_prefix"  #!!!
    )

    # Безопасное добавление документов в векторное хранилище
    safe_add_documents(vector_store, all_chunks)

Добавление в Chroma:   0%|          | 0/105 [00:00<?, ?doc/s]


Вычисление эмбеддингов: 100%|██████████| 14/14 [01:05<00:00,  4.68s/batch]


Все документы успешно добавлены!


In [ ]:
if test_e5_with_prefix:
    !zip -r chroma_db_prefix.zip chroma_db_prefix/

  adding: chroma_db_prefix/ (stored 0%)
  adding: chroma_db_prefix/9156f6c1-323a-4a45-90b4-2283f3581fbd/ (stored 0%)
  adding: chroma_db_prefix/9156f6c1-323a-4a45-90b4-2283f3581fbd/link_lists.bin (stored 0%)
  adding: chroma_db_prefix/9156f6c1-323a-4a45-90b4-2283f3581fbd/header.bin (deflated 61%)
  adding: chroma_db_prefix/9156f6c1-323a-4a45-90b4-2283f3581fbd/length.bin (deflated 100%)
  adding: chroma_db_prefix/9156f6c1-323a-4a45-90b4-2283f3581fbd/data_level0.bin (deflated 100%)
  adding: chroma_db_prefix/chroma.sqlite3 (deflated 52%)


# Show the output

In [ ]:
vector_store = Chroma(
    collection_name="docs",
    embedding_function=embedder,
    persist_directory="./chroma_db_prefix"
)

In [ ]:
k = 10

bm25_retriever = BM25Retriever.from_documents(
    all_chunks,
    preprocess_func=preprocess
)
bm25_retriever.k = k

vector_retriever = vector_store.as_retriever(search_kwargs={"k": k})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.35, 0.65],
)

In [ ]:
# pickle.dump(bm25_retriever, open('retriever', 'wb'))

In [ ]:
preprocess('query: Как дела?')

['дел']

что-то меня смущает, как предобрабатывается текст, раньше писал ['как', 'дел']

In [ ]:
query = "Как поступить попасть в пропком?"
results = ensemble_retriever.get_relevant_documents(query)

Вычисление эмбеддингов: 100%|██████████| 1/1 [00:00<00:00,  1.11batch/s]


In [ ]:
results[:3]

[Document(id='22599d9f-f0cd-4c3e-834f-ed5ff699e69e', metadata={'source': ' Как подать жалобу?'}, page_content='Как подать жалобу?. Кнопка в приложении (https://app.profcomff.com/apps/6)'),
 Document(id='74993772-9cf2-43a0-9df3-b3766e4e7203', metadata={'source': 'Подготовка к пересдаче'}, page_content='Подготовка к пересдаче. Как подготовиться к пересдаче? Чтобы подготовиться к пересдачам, можно прослушать лекционные и семинарские курсы на портале teach-in.ru'),
 Document(id='0d362e60-1217-4fd1-9ba9-74ca2c2e5c2b', metadata={'source': 'Запись на МФК'}, page_content='Запись на МФК. Как записаться на МФК? Запись на МФК осуществляется через портал lk.msu.ru.')]